<a href="https://colab.research.google.com/github/dannial-ahmad/my-projects/blob/main/USD_PKR_Dollar_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💱 USD/PKR Dollar Rate Level Advisor
**Colab + Ollama + Gradio (Explainable AI)**

1. Transparent scoring engine: every factor has a direction and weight, and its contribution is visible.
2. Ollama LLM: turns the computed contributions into a plain-language explanation and buy/sell suggestion.

Use **Runtime → Change runtime type → T4 GPU** for faster answers. Run the cells in order.

### Cell 1: Install dependencies

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
!pip -q install "gradio>=5,<6" requests matplotlib pandas
!curl -fsSL https://ollama.com/install.sh | sh

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.13 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you hav

### Cell 2: Start the Ollama server and pull a model

In [ ]:
import subprocess, time, requests, os

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "llama3.2:3b"   # alternatives: "qwen2.5:3b", "phi3:mini", "llama3.1:8b" (needs GPU)

# Start server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})

# Wait until it responds
for _ in range(60):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print("✅ Ollama server is running")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start")

!ollama pull {MODEL_NAME}

✅ Ollama server is running



### Cell 3: Factor definitions and XAI scoring engine

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CHOICES = ["Increasing", "Stable", "Decreasing"]
STATE_VALUE = {"Increasing": 1, "Stable": 0, "Decreasing": -1}

# sign = +1 -> if the factor INCREASES, USD/PKR tends to go UP (rupee weaker)
# sign = -1 -> if the factor INCREASES, USD/PKR tends to go DOWN (rupee stronger)
FACTORS = [
    # key, label, group, sign, weight, explanation
    ("exports",        "Exports",                          "inflow", -1, 1.0, "More exports bring more dollars into Pakistan."),
    ("remittances",    "Remittances",                      "inflow", -1, 1.2, "Overseas workers' transfers add foreign currency supply."),
    ("fdi",            "Foreign investment",               "inflow", -1, 1.0, "Foreign investment brings dollars into the country."),
    ("ext_financing",  "External financing / inflows",     "inflow", -1, 1.0, "Loans, deposits and aid increase dollar supply."),
    ("reserves",       "FX reserves",                      "inflow", -1, 1.2, "Higher reserves reduce pressure on the rupee."),
    ("current_acct",   "Current account (improving)",      "inflow", -1, 1.1, "An improving current account means a lower need for dollars."),

    ("imports",        "Imports",                          "demand", +1, 1.3, "Importers need more dollars to pay foreign suppliers."),
    ("oil",            "Oil prices",                       "demand", +1, 1.3, "Higher oil prices raise the dollar energy import bill."),
    ("outflows",       "Foreign investors withdrawing",    "demand", +1, 1.1, "Capital outflows increase demand for dollars."),
    ("dollar_demand",  "General demand for the dollar",    "demand", +1, 1.0, "Higher demand pushes the price of the dollar up."),

    ("inflation",      "Inflation",                        "macro",  +1, 0.9, "Persistent inflation erodes rupee purchasing power."),
    ("policy_rate",    "SBP policy rate (Increasing = hike)", "macro", -1, 0.8, "Higher rates can attract inflows and support the rupee."),
    ("political",      "Political / economic uncertainty", "macro",  +1, 1.2, "Uncertainty makes people hold dollars and scares investors."),
]

def compute_score(states: dict):
    rows, total_w, total_c, total_abs = [], 0.0, 0.0, 0.0
    for key, label, group, sign, w, why in FACTORS:
        choice = states.get(key, "Stable")
        contrib = w * sign * STATE_VALUE[choice]
        total_w += w
        total_c += contrib
        total_abs += abs(contrib)
        effect = "Pushes dollar UP ⬆" if contrib > 0 else "Pushes dollar DOWN ⬇" if contrib < 0 else "Neutral"
        rows.append({"Factor": label, "Your input": choice, "Effect": effect,
                     "Contribution": round(contrib, 2), "Why": why})
    raw = total_c / total_w                          # range -1 .. +1
    pressure = float(np.clip(50 + 60 * raw, 0, 100)) # 0 = strongly falling, 100 = strongly rising
    agreement = (abs(total_c) / total_abs * 100) if total_abs > 0 else 0.0
    if pressure >= 60:
        level = "HIGH"
    elif pressure <= 40:
        level = "LOW"
    else:
        level = "STABLE / NEUTRAL"
    df = pd.DataFrame(rows).sort_values("Contribution", key=abs, ascending=False).reset_index(drop=True)
    return df, pressure, level, agreement

def make_plot(df):
    d = df[df["Contribution"] != 0].sort_values("Contribution")
    fig, ax = plt.subplots(figsize=(7, max(3, 0.45 * len(d) + 1)))
    if d.empty:
        ax.text(0.5, 0.5, "All factors are 'Stable'", ha="center", va="center", fontsize=12)
        ax.axis("off")
    else:
        colors = ["#ef4444" if v > 0 else "#10b981" for v in d["Contribution"]]
        ax.barh(d["Factor"], d["Contribution"], color=colors)
        ax.axvline(0, color="#64748b", lw=1)
        ax.set_xlabel("← pushes dollar DOWN        pushes dollar UP →")
        ax.set_title("Why the model says so: factor contributions", fontsize=11, fontweight="bold")
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    fig.tight_layout()
    return fig

### Cell 4: Ollama explanation layer

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful financial-education assistant for Pakistan's USD/PKR market. "
    "You explain a pre-computed factor analysis in simple language. "
    "Use ONLY the factors and numbers provided. Never invent exchange-rate numbers, news, or statistics. "
    "Be honest about uncertainty. You are not a financial advisor."
)

def ask_ollama(prompt, model=MODEL_NAME):
    try:
        r = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json={"model": model,
                  "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                               {"role": "user", "content": prompt}],
                  "stream": False,
                  "options": {"temperature": 0.3}},
            timeout=300)
        r.raise_for_status()
        return r.json()["message"]["content"]
    except Exception as e:
        return f"⚠️ Could not reach the Ollama model ({e}). The score above is still valid."

def build_prompt(df, pressure, level, agreement, intent, horizon, current_rate, notes):
    up = df[df["Contribution"] > 0]
    down = df[df["Contribution"] < 0]
    fmt = lambda d: "\n".join(f"- {r.Factor} ({r._2}): {r.Why}" for r in d.itertuples()) or "- none"
    return f"""
Analysis result:
- Dollar pressure score: {pressure:.0f}/100 (0 = dollar likely falling, 100 = dollar likely rising)
- Expected dollar rate level: {level}
- Signal agreement among factors: {agreement:.0f}%
- User wants to: {intent}
- Time horizon: {horizon}
- Current open-market rate entered by user: {current_rate or "not provided"}
- User's political/news notes: {notes or "none"}

Factors pushing the dollar UP:
{fmt(up)}

Factors pushing the dollar DOWN:
{fmt(down)}

Write the answer with these exact headings:
**1. Why the dollar may be {level.lower()}**  (3-4 sentences, cite the top factors)
**2. What this means for you ({intent})**  (clear suggestion: act now, wait, or split into parts, with reasons)
**3. Key risks**  (2-3 bullet points, include anything from the user's notes)
Keep it under 220 words and end with one line: "This is educational analysis, not financial advice."
""".strip()

### Cell 5: Main prediction function

In [ ]:
def level_card(level, pressure, agreement):
    color = {"HIGH": "#ef4444", "LOW": "#10b981"}.get(level, "#f59e0b")
    icon = {"HIGH": "📈", "LOW": "📉"}.get(level, "➖")
    return f"""
    <div style="border-radius:16px;padding:22px;background:linear-gradient(135deg,{color}22,{color}08);
                border:2px solid {color};text-align:center;">
      <div style="font-size:15px;opacity:.75;">Expected USD/PKR level</div>
      <div style="font-size:38px;font-weight:800;color:{color};margin:4px 0;">{icon} {level}</div>
      <div style="font-size:14px;opacity:.85;">Dollar pressure score: <b>{pressure:.0f}/100</b> &nbsp;|&nbsp;
      Signal agreement: <b>{agreement:.0f}%</b></div>
      <div style="height:10px;background:#e2e8f0;border-radius:99px;margin-top:14px;overflow:hidden;">
        <div style="width:{pressure:.0f}%;height:100%;background:{color};"></div></div>
    </div>"""

def quick_advice(level, intent):
    if intent == "Buy dollars":
        return {"HIGH": "🟠 Dollar likely to rise: consider buying soon or in 2-3 instalments.",
                "LOW": "🟢 Dollar likely to ease: consider waiting or buying gradually.",
                }.get(level, "🟡 No strong signal: buy only what you need, in small parts.")
    return {"HIGH": "🟢 Dollar likely strong: a good window to sell, or sell in parts.",
            "LOW": "🟠 Dollar likely to weaken: sell sooner if you need rupees, don't wait.",
            }.get(level, "🟡 No strong signal: sell in small parts rather than all at once.")

def analyze(intent, horizon, current_rate, notes, model, *factor_values):
    states = {f[0]: v for f, v in zip(FACTORS, factor_values)}
    df, pressure, level, agreement = compute_score(states)
    card = level_card(level, pressure, agreement)
    fig = make_plot(df)
    table = df[["Factor", "Your input", "Effect", "Contribution"]]
    prompt = build_prompt(df, pressure, level, agreement, intent, horizon, current_rate, notes)
    explanation = ask_ollama(prompt, model or MODEL_NAME)
    return card, quick_advice(level, intent), fig, table, explanation

### Cell 6: Gradio interface (run last, then open the public link)

In [ ]:
import gradio as gr

theme = gr.themes.Soft(
    primary_hue="emerald", secondary_hue="teal", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
).set(button_primary_background_fill="linear-gradient(90deg,#059669,#0d9488)",
      button_primary_text_color="white")

css = """
.hero {text-align:center;padding:26px;border-radius:18px;color:white;
       background:linear-gradient(135deg,#064e3b,#0f766e 60%,#0891b2);margin-bottom:10px;}
.hero h1 {margin:0;font-size:30px;color:white;} .hero p {margin:6px 0 0;opacity:.9;}
footer {display:none !important;}
"""

GROUPS = {
    "inflow": "💵 Dollar INFLOWS (supply of dollars)",
    "demand": "🛒 Dollar DEMAND (outflows)",
    "macro":  "🏛️ Macro, policy & politics",
}

with gr.Blocks(theme=theme, css=css, title="USD/PKR Level Advisor") as demo:
    gr.HTML("""<div class="hero"><h1>💱 USD/PKR Dollar Level Advisor</h1>
    <p>Explainable AI: tells you whether the dollar is likely HIGH or LOW, and exactly why.</p></div>""")

    with gr.Row():
        with gr.Column(scale=5):
            with gr.Group():
                intent = gr.Radio(["Buy dollars", "Sell dollars"], value="Buy dollars", label="I want to")
                horizon = gr.Dropdown(["1 week", "1 month", "3 months", "6 months"], value="1 month", label="Time horizon")
                current_rate = gr.Textbox(label="Current open-market rate (optional)", placeholder="e.g. 280")
                notes = gr.Textbox(label="Political / economic news notes (optional)", lines=3,
                                   placeholder="e.g. IMF review next week, budget announcement, protests...")
                model = gr.Textbox(label="Ollama model", value=MODEL_NAME)

            dropdowns = []
            for gkey, gtitle in GROUPS.items():
                with gr.Accordion(gtitle, open=True):
                    for key, label, group, sign, w, why in FACTORS:
                        if group == gkey:
                            dropdowns.append(gr.Dropdown(CHOICES, value="Stable", label=label, info=why))
            btn = gr.Button("🔍 Analyze Dollar Level", variant="primary", size="lg")

        with gr.Column(scale=6):
            card = gr.HTML()
            advice = gr.Markdown()
            plot = gr.Plot(label="Factor contributions (XAI)")
            with gr.Accordion("📋 Detailed factor table", open=False):
                table = gr.Dataframe(interactive=False)
            gr.Markdown("### 🧠 AI explanation (Ollama)")
            explanation = gr.Markdown()

    # dropdown order must match FACTORS order, so rebuild the FACTORS order from groups
    ordered_factors = [f for g in GROUPS for f in FACTORS if f[2] == g]
    def analyze_ordered(intent, horizon, current_rate, notes, model, *vals):
        by_key = {f[0]: v for f, v in zip(ordered_factors, vals)}
        return analyze(intent, horizon, current_rate, notes, model, *[by_key[f[0]] for f in FACTORS])

    btn.click(analyze_ordered,
              inputs=[intent, horizon, current_rate, notes, model] + dropdowns,
              outputs=[card, advice, plot, table, explanation])

    gr.Markdown("<center><small>⚠️ Educational tool only. Not financial advice. "
                "Inputs are your own judgement of each factor's current trend.</small></center>")

demo.launch(share=True, debug=False)

/tmp/ipykernel_509/973849993.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css, title="USD/PKR Level Advisor") as demo:
/tmp/ipykernel_509/973849993.py:22: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css, title="USD/PKR Level Advisor") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://209a47e66e2825ea0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
